# Батчеве перефразування відгуків за допомогою NLLB

Використовуємо модель **facebook/nllb-200-distilled-600M** для перефразування через back-translation (укр→англ→укр)

**Мета:** Розширити існуючі N відгуків до 5000 шляхом перефразування існуючих

In [ ]:
# Вимкнення зайвих попереджень
from transformers.utils import logging
logging.set_verbosity_error()

In [ ]:
# Завантаження моделі NLLB
from transformers import pipeline
import torch

print("🔄 Завантаження моделі facebook/nllb-200-distilled-600M...")
print(f"🎮 CUDA доступна: {torch.cuda.is_available()}")

translator = pipeline(
    task="translation",
    model="facebook/nllb-200-distilled-600M",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="cuda" if torch.cuda.is_available() else "cpu"
)

print("Модель завантажено")

## Тест на одному відгуку

Перевіряємо якість перефразування

In [ ]:
# Тестовий відгук
test_text = """Надзвичайно задоволений візитом до цього лікаря-уролога! Неймовірно професійний підхід, фантастичне лікування та ідеальна увага до пацієнта. Найкращий лікар, якого я коли-небудь зустрічав, надав мені найкращу допомогу під час післяопераційного огляду."""

print("ОРИГІНАЛ:")
print(test_text)
print("\n" + "="*80 + "\n")

# Українська → Англійська
text_en = translator(test_text, src_lang="ukr_Cyrl", tgt_lang="eng_Latn")
print("ПЕРЕКЛАД (УКР → ENG):")
print(text_en[0]['translation_text'])
print("\n" + "="*80 + "\n")

# Англійська → Українська (перефразування)
text_paraphrased = translator(text_en[0]['translation_text'], src_lang="eng_Latn", tgt_lang="ukr_Cyrl")
print("ПЕРЕФРАЗОВАНЕ (ENG → УКР):")
print(text_paraphrased[0]['translation_text'])

---

# Батчеве перефразування для досягнення 5000 відгуків

Завантажуємо існуючі відгуки і циклічно перефразовуємо їх батчами

In [ ]:
# Імпорти для батчевої обробки
import pandas as pd
from tqdm.notebook import tqdm
import time
import random

# Конфігурація
INPUT_FILE = "../my-crawler/fake-reviews_copy.csv"
OUTPUT_FILE = "../my-crawler/fake-reviews_copy.csv"
TARGET_COUNT = 5000
BATCH_SIZE = 8  # Обробка по 8 відгуків одночасно

print(f"📁 Файл: {INPUT_FILE}")
print(f"🎯 Цільова кількість: {TARGET_COUNT}")
print(f"📦 Розмір батчу: {BATCH_SIZE}")

In [ ]:
# Функція для батчевого перефразування з варіативністю
def paraphrase_batch(texts, add_variation=True):
    """
    Перефразування батчу текстів через back-translation
    
    add_variation: додає випадковість для різноманітності результатів
    """
    # Випадкові параметри для варіативності (кожен виклик буде давати різні результати)
    if add_variation:
        # Варіюємо num_beams (від 2 до 5)
        num_beams = random.randint(2, 5)
        # Варіюємо temperature (від 0.8 до 1.2)
        temperature = random.uniform(0.8, 1.2)
        # Варіюємо top_p
        top_p = random.uniform(0.85, 0.98)
        
        generation_kwargs = {
            "num_beams": num_beams,
            "do_sample": True,
            "temperature": temperature,
            "top_p": top_p,
        }
    else:
        generation_kwargs = {}
    
    # Українська → Англійська
    translated = translator(
        texts,
        src_lang="ukr_Cyrl",
        tgt_lang="eng_Latn",
        batch_size=BATCH_SIZE,
        **generation_kwargs
    )
    english_texts = [t['translation_text'] for t in translated]
    
    # Генеруємо нові параметри для зворотного перекладу (ще більше варіативності)
    if add_variation:
        num_beams = random.randint(2, 5)
        temperature = random.uniform(0.8, 1.2)
        top_p = random.uniform(0.85, 0.98)
        
        generation_kwargs = {
            "num_beams": num_beams,
            "do_sample": True,
            "temperature": temperature,
            "top_p": top_p,
        }
    
    # Англійська → Українська
    paraphrased = translator(
        english_texts,
        src_lang="eng_Latn",
        tgt_lang="ukr_Cyrl",
        batch_size=BATCH_SIZE,
        **generation_kwargs
    )
    
    return [p['translation_text'] for p in paraphrased]

print("Функція paraphrase_batch() готова")
print("Варіативність: num_beams (2-5), temperature (0.8-1.2), top_p (0.85-0.98)")

In [ ]:
# Завантаження існуючих відгуків
print(f"Читання {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)
current_count = len(df)
print(f"✓ Завантажено {current_count} відгуків")

if current_count >= TARGET_COUNT:
    print(f"Вже є {current_count} відгуків (ціль: {TARGET_COUNT})")
else:
    needed = TARGET_COUNT - current_count
    print(f"\nПотрібно згенерувати ще {needed} відгуків")
    print(f"Це ~{needed / current_count:.1f} циклів по оригінальним відгукам")

# Показуємо приклад
print("\nПриклад даних:")
df.head()

## Тест батчевого перефразування

Перевіряємо на 3 прикладах перед повним запуском.

**Варіативність забезпечується через:**
- Випадковий `num_beams` (2-5) - різна стратегія пошуку
- Випадковий `temperature` (0.8-1.2) - різний рівень креативності
- Випадковий `top_p` (0.85-0.98) - різна вибірковість токенів

Це означає, що навіть той самий текст буде перефразовуватися по-різному кожного разу! 🎲

In [ ]:
# Тестування на 3 перших відгуках
test_reviews = df.head(3)['text'].tolist()

print("ОРИГІНАЛИ:")
for i, text in enumerate(test_reviews, 1):
    print(f"\n{i}. {text[:100]}...")

print("\n" + "="*80 + "\n")
print("Перефразовуємо...\n")

paraphrased_test = paraphrase_batch(test_reviews)

print("ПЕРЕФРАЗОВАНІ:")
for i, text in enumerate(paraphrased_test, 1):
    print(f"\n{i}. {text[:100]}...")

# Додатковий тест - показати що той самий текст перефразовується по-різному
print("\n" + "="*80 + "\n")
print("ТЕСТ ВАРІАТИВНОСТІ (той самий текст 3 рази):\n")
sample_text = test_reviews[0]
print(f"ОРИГІНАЛ:\n{sample_text[:150]}...\n")

for i in range(3):
    varied = paraphrase_batch([sample_text])[0]
    print(f"Варіант {i+1}:\n{varied[:150]}...\n")

## ОСНОВНИЙ ЦИКЛ ГЕНЕРАЦІЇ


In [ ]:
# Основний цикл генерації
if current_count < TARGET_COUNT:
    start_time = time.time()
    
    # Оригінальні відгуки для циклічного перефразування
    original_reviews = df.to_dict('records')
    
    # Генерація нових відгуків
    generated = 0
    cycle = 0
    next_id = current_count + 1
    
    with tqdm(total=needed, desc="Генерація відгуків") as pbar:
        while generated < needed:
            cycle += 1
            print(f"\n🔄 Цикл #{cycle}")
            
            # Проходимо по оригінальним відгукам батчами
            for i in range(0, len(original_reviews), BATCH_SIZE):
                if generated >= needed:
                    break
                
                # Беремо батч
                batch = original_reviews[i:i+BATCH_SIZE]
                texts = [review['text'] for review in batch]
                
                # Перефразовуємо
                try:
                    paraphrased = paraphrase_batch(texts)
                    
                    # Створюємо нові записи
                    new_rows = []
                    for j, (original_review, paraphrased_text) in enumerate(zip(batch, paraphrased)):
                        new_row = {
                            'id': next_id,
                            'text': paraphrased_text,
                            'strategy': original_review['strategy'],
                            'isFake': True
                        }
                        new_rows.append(new_row)
                        next_id += 1
                        generated += 1
                        pbar.update(1)
                        
                        if generated >= needed:
                            break
                    
                    # Додаємо до DataFrame
                    df_new = pd.DataFrame(new_rows)
                    df = pd.concat([df, df_new], ignore_index=True)
                    
                    # Зберігаємо після кожного батчу (для збереження прогресу)
                    df.to_csv(OUTPUT_FILE, index=False)
                    
                except Exception as e:
                    print(f"\nПомилка на батчі: {e}")
                    print("Очікування 5 секунд...")
                    time.sleep(5)
                    continue
    
    elapsed = time.time() - start_time
    print(f"\nГотово")
    print(f"Згенеровано {generated} нових відгуків за {cycle} циклів")
    print(f"Всього відгуків: {len(df)}")
    print(f"Час виконання: {elapsed/60:.1f} хвилин ({elapsed/generated:.2f} сек/відгук)")
    print(f"Збережено в {OUTPUT_FILE}")
else:
    print("Вже досягнуто цільової кількості")

## Фінальна статистика

In [ ]:
# Перевірка фінальних результатів
print(f"Фінальна статистика:")
print(f"Всього відгуків: {len(df)}")
print(f"\nРозподіл по стратегіях:")
print(df['strategy'].value_counts())
print(f"\nОстанні 5 відгуків:")
df.tail()